<a href="https://www.kaggle.com/code/simarbirsinghsandhu/xgboost-gpu-fe-encoding?scriptVersionId=315895961" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 🏎️ F1 Pit Stop Prediction | XGBoost + Optuna GPU Baseline
### Playground Series S6E5

**🏆 OOF ROC-AUC: 0.957087 | 📊 Public LB: 0.94925**

⚡ XGBoost + Optuna (GPU) | 5-Fold Stratified CV

**What makes this baseline strong:**
- Reconstructed `Normalized_TyreLife` — the column intentionally removed by the host
- Original dataset incorporated with domain gap flag
- Optuna-tuned directly on ROC-AUC

> Ready to build on — upvote if useful 🙌 | [Full Detailed EDA](https://www.kaggle.com/code/simarbirsinghsandhu/detailed-eda-on-comp-original-dataset) | [CatBoost GPU for ensembling](https://www.kaggle.com/code/simarbirsinghsandhu/catboost-gpu-fe-encodin)

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
import matplotlib.pyplot as plt
import warnings

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
print(f"xgboost {xgb.__version__}")

## 2. Config

In [ ]:
CONFIG = {
    "train_path"       : "/kaggle/input/competitions/playground-series-s6e5/train.csv",
    "test_path"        : "/kaggle/input/competitions/playground-series-s6e5/test.csv",
    "orig_path"        : "/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv",
    "target_col"       : "PitNextLap",
    "id_col"           : "id",
    "n_folds"          : 5,
    "seed"             : 42,
    "optuna_trials"    : 50,
    "optuna_folds"     : 3,
    "optuna_subsample" : 200_000,
}

## 3. Load Data

In [ ]:
train = pd.read_csv(CONFIG["train_path"])
test  = pd.read_csv(CONFIG["test_path"])
orig  = pd.read_csv(CONFIG["orig_path"])

orig.drop(columns=["Normalized_TyreLife"], inplace=True, errors="ignore")

print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")
print(f"Orig  : {orig.shape}")
print(f"\nTarget distribution:")
print(train["PitNextLap"].value_counts(normalize=True).round(4))

## 4. Feature Engineering

Based on EDA findings — three core signals drive pit stops:
1. **Tyre age** relative to compound lifecycle
2. **Race timing** — when in the race the lap occurs  
3. **Degradation rate** — how fast the tyre is wearing per lap

We reconstruct `Normalized_TyreLife` which was intentionally removed from the competition data. Using compound-specific median stint lengths from EDA: Soft=14, Medium=17, Hard=23.

In [ ]:
COMPOUND_STINT_MEDIANS = {
    "SOFT"        : 14.0,
    "MEDIUM"      : 17.0,
    "HARD"        : 23.0,
    "INTERMEDIATE": 16.0,
    "WET"         : 14.0,
}

COMPOUND_HARDNESS = {
    "SOFT"        : 1,
    "MEDIUM"      : 2,
    "HARD"        : 3,
    "INTERMEDIATE": 1,
    "WET"         : 0,
}

def engineer_features(df):
    df = df.copy()

    # ── Reconstructed Normalized TyreLife ────────────────────
    # Approximates the removed column: TyreLife / expected_stint_length
    df["ExpectedStint"]       = df["Compound"].map(COMPOUND_STINT_MEDIANS).fillna(17.0)
    df["TyreLife_Normalized"] = df["TyreLife"] / df["ExpectedStint"]

    # ── Tyre age transforms ───────────────────────────────────
    df["TyreLife_sq"]         = df["TyreLife"] ** 2
    df["TyreLife_sqrt"]       = np.sqrt(df["TyreLife"])
    df["TyreLife_log1p"]      = np.log1p(df["TyreLife"])

    # ── Compound hardness + interaction ──────────────────────
    df["Compound_Hardness"]   = df["Compound"].map(COMPOUND_HARDNESS).fillna(2).astype(int)
    df["TyreLife_x_Hardness"] = df["TyreLife"] * df["Compound_Hardness"]
    df["Norm_x_Hardness"]     = df["TyreLife_Normalized"] * df["Compound_Hardness"]

    # ── Tyre age threshold flags ──────────────────────────────
    df["Is_Fresh"]            = (df["TyreLife"] <= 3).astype(np.int8)
    df["Is_Old"]              = (df["TyreLife"] > 20).astype(np.int8)
    df["Is_VeryOld"]          = (df["TyreLife"] > 40).astype(np.int8)
    df["Is_FirstStint"]       = (df["Stint"] == 1).astype(np.int8)

    # ── Degradation rate ──────────────────────────────────────
    df["DegRate"]             = df["Cumulative_Degradation"] / (df["TyreLife"] + 1)

    # ── Race stage flags ──────────────────────────────────────
    df["Early_Race"]          = (df["RaceProgress"] < 0.25).astype(np.int8)
    df["Late_Race"]           = (df["RaceProgress"] >= 0.75).astype(np.int8)
    df["VeryLate_Race"]       = (df["RaceProgress"] >= 0.90).astype(np.int8)

    # ── 2025 corruption flag ──────────────────────────────────
    df["Is_2025"]             = (df["Year"] == 2025).astype(np.int8)

    return df

train = engineer_features(train)
test  = engineer_features(test)
orig  = engineer_features(orig)

new_feats = ["TyreLife_Normalized", "TyreLife_sq", "TyreLife_sqrt",
             "TyreLife_log1p", "Compound_Hardness", "TyreLife_x_Hardness",
             "Norm_x_Hardness", "Is_Fresh", "Is_Old", "Is_VeryOld",
             "Is_FirstStint", "DegRate", "Early_Race", "Late_Race",
             "VeryLate_Race", "Is_2025"]
print(f"Engineered {len(new_feats)} new features")

## 5. Preprocessing

Label-encode the three categorical columns (Driver, Compound, Race).  
Fit on train + test + original combined so no unseen labels at inference.

In [ ]:
CAT_COLS  = ["Driver", "Compound", "Race"]
DROP_COLS = ["id", "PitNextLap", "ExpectedStint"]

encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col], orig[col]],
                         ignore_index=True).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))
    orig[col]  = le.transform(orig[col].astype(str))
    encoders[col] = le

# Add is_original flag — lets model learn domain gap
train["is_original"] = 0
orig["is_original"]  = 1
test["is_original"]  = 0

FEATURE_COLS = [c for c in train.columns if c not in DROP_COLS]

# Align original to train feature set
orig_aligned = orig.reindex(columns=FEATURE_COLS + ["PitNextLap"], fill_value=0)
orig_aligned["PitNextLap"] = orig_aligned["PitNextLap"].astype(float)

## 6. Encoding

In [ ]:
def add_static_encodings(df):
    df = df.copy()

    for comp in ['SOFT', 'MEDIUM', 'HARD', 'INTERMEDIATE', 'WET']:
        df[f'Is_{comp}'] = (df['Compound_orig'] == comp).astype(np.int8)
    
    for s in [1, 2, 3, 4]:
        df[f'Is_Stint{s}'] = (df['Stint'] == s).astype(np.int8)
    
    for yr in [2022, 2023, 2024, 2025]:
        df[f'Is_{yr}'] = (df['Year'] == yr).astype(np.int8)
    
    df['LapDelta_sq']             = df['LapTime_Delta'] ** 2
    df['LapDelta_abs']            = df['LapTime_Delta'].abs()
    df['LapDelta_extreme']        = (df['LapTime_Delta'].abs() > 10).astype(np.int8)
    df['RaceProgress_x_TyreLife'] = df['RaceProgress'] * df['TyreLife']
    df['RaceProgress_x_Norm']     = df['RaceProgress'] * df['TyreLife_Normalized']
    df['Late_x_OldTyre']          = df['Late_Race'] * df['Is_Old']
    df['VeryLate_x_VeryOld']      = df['VeryLate_Race'] * df['Is_VeryOld']
    df['Stint_x_Normalized']      = df['Stint'] * df['TyreLife_Normalized']
    df['Stint2_x_Old']            = ((df['Stint'] == 2) & (df['Is_Old'] == 1)).astype(np.int8)
    df['DegRate_x_Hardness']      = df['DegRate'] * df['Compound_Hardness']
    df['Laps_Until_Due']          = df['ExpectedStint'] - df['TyreLife']
    df['Overdue']                 = (df['Laps_Until_Due'] < 0).astype(np.int8)
    df['Position_x_TyreLife']     = df['Position'] * df['TyreLife']
    df['Is_2023']                 = (df['Year'] == 2023).astype(np.int8)
    
    return df

# Need original compound strings before label encoding for one-hot
# So add a column BEFORE label encoding
train['Compound_orig'] = encoders['Compound'].inverse_transform(train['Compound'])
test['Compound_orig']  = encoders['Compound'].inverse_transform(test['Compound'])
orig['Compound_orig']  = encoders['Compound'].inverse_transform(orig['Compound'])

train = add_static_encodings(train)
test  = add_static_encodings(test)
orig  = add_static_encodings(orig)

# Drop helper column
train.drop(columns=['Compound_orig'], inplace=True)
test.drop(columns=['Compound_orig'],  inplace=True)
orig.drop(columns=['Compound_orig'],  inplace=True)

# Rebuild train_full and arrays
DROP_COLS = ["id", "PitNextLap", "ExpectedStint"]
orig_aligned = orig.reindex(columns=[c for c in train.columns if c not in DROP_COLS] + ["PitNextLap"], fill_value=0)
orig_aligned["PitNextLap"] = orig_aligned["PitNextLap"].astype(float)
train_full = pd.concat([train, orig_aligned], ignore_index=True)

FEATURE_COLS = [c for c in train_full.columns if c not in DROP_COLS]
print(f"Total features before target encoding: {len(FEATURE_COLS)}")

In [ ]:
train_full = pd.concat([train, orig_aligned], ignore_index=True)

X      = train_full[FEATURE_COLS].values
y      = train_full["PitNextLap"].values
X_test = test[FEATURE_COLS].values

print(f"Features    : {len(FEATURE_COLS)}")
print(f"Train rows  : {len(X):,}  (synthetic + original)")
print(f"Positive rate: {y.mean():.4f}")
print(f"\nFeature list:\n{FEATURE_COLS}")

## 7. Optuna Hyperparameter Search

3-fold CV on 200K subsample — fast enough to run 50 trials in ~10 minutes on GPU.  
Optimizing directly for ROC-AUC, which is the competition metric.

Skip this cell on reruns and use the saved best params below.

In [ ]:
def run_optuna(X, y, cfg):
    n_sub = cfg["optuna_subsample"]
    rng   = np.random.RandomState(cfg["seed"])
    idx   = rng.choice(len(X), min(n_sub, len(X)), replace=False)
    X_opt = X[idx]
    y_opt = y[idx]
    print(f"Subsampled to {len(X_opt):,} rows for Optuna")

    kf = StratifiedKFold(n_splits=cfg["optuna_folds"],
                         shuffle=True, random_state=cfg["seed"])

    def objective(trial):
        params = {
            "objective"        : "binary:logistic",
            "eval_metric"      : "auc",
            "device"           : "cuda",
            "tree_method"      : "hist",
            "n_estimators"     : 5000,
            "verbosity"        : 0,
            "random_state"     : cfg["seed"],
            "learning_rate"    : trial.suggest_float("learning_rate",    0.005, 0.05,  log=True),
            "max_depth"        : trial.suggest_int(  "max_depth",        4,     8),
            "min_child_weight" : trial.suggest_int(  "min_child_weight", 1,     20),
            "gamma"            : trial.suggest_float("gamma",            0.0,   2.0),
            "subsample"        : trial.suggest_float("subsample",        0.6,   1.0),
            "colsample_bytree" : trial.suggest_float("colsample_bytree", 0.5,   1.0),
            "reg_lambda"       : trial.suggest_float("reg_lambda",       0.01,  10.0, log=True),
            "reg_alpha"        : trial.suggest_float("reg_alpha",        0.01,  10.0, log=True),
            "scale_pos_weight" : trial.suggest_float("scale_pos_weight", 1.0,   8.0),
        }

        aucs = []
        for tr_idx, va_idx in kf.split(X_opt, y_opt):
            model = xgb.XGBClassifier(
                **params,
                callbacks=[xgb.callback.EarlyStopping(
                    rounds=100, metric_name="auc",
                    maximize=True, save_best=True
                )]
            )
            model.fit(X_opt[tr_idx], y_opt[tr_idx],
                      eval_set=[(X_opt[va_idx], y_opt[va_idx])],
                      verbose=False)
            aucs.append(roc_auc_score(y_opt[va_idx],
                                      model.predict_proba(X_opt[va_idx])[:, 1]))
        return np.mean(aucs)

    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=cfg["seed"]))
    study.optimize(objective, n_trials=cfg["optuna_trials"], show_progress_bar=True)

    print(f"\n✅ Best ROC-AUC : {study.best_value:.6f}")
    print(f"Best params    :\n{study.best_params}")
    return study.best_params, study

# Uncomment to run optuna tune
# best_params, study = run_optuna(X, y, CONFIG)

## 8. Best params
These are auto-filled by Optuna above. If you want to skip Optuna on re-runs, comment out the cell above and paste your best params here.

In [ ]:
# best_params is already set by Optuna above
# If you skipped Optuna, uncomment and paste your params

best_params = {
    'learning_rate': 0.03165494943023957,
    'max_depth': 8,
    'min_child_weight': 2,
    'gamma': 0.9848620716745634, 
    'subsample': 0.695355115905674, 
    'colsample_bytree': 0.5062336804817325,
    'reg_lambda': 2.307646503783304, 
    'reg_alpha': 1.7481823118274633, 
    'scale_pos_weight': 2.0951358948997787
}

print("Using params:", best_params)

## 9. OOF Training

Full 5-fold stratified CV on the complete training set (synthetic + original).  
OOF predictions are used to compute a reliable CV score before submission.

In [ ]:
def xgb_oof(X, y, X_test, params, cfg):
    kf         = StratifiedKFold(n_splits=cfg["n_folds"],
                                 shuffle=True, random_state=cfg["seed"])
    oof_preds  = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    fold_aucs  = []

    run_params = {
        "objective"       : "binary:logistic",
        "eval_metric"     : "auc",
        "device"          : "cuda",
        "tree_method"     : "hist",
        "n_estimators"    : 5000,
        "verbosity"       : 0,
        "random_state"    : cfg["seed"],
        **params,
    }

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X, y)):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        model = xgb.XGBClassifier(
            **run_params,
            callbacks=[xgb.callback.EarlyStopping(
                rounds=100, metric_name="auc",
                maximize=True, save_best=True
            )]
        )
        model.fit(X_tr, y_tr,
                  eval_set=[(X_va, y_va)],
                  verbose=200)

        val_preds             = model.predict_proba(X_va)[:, 1]
        oof_preds[va_idx]     = val_preds
        test_preds           += model.predict_proba(X_test)[:, 1] / cfg["n_folds"]
        auc                   = roc_auc_score(y_va, val_preds)
        fold_aucs.append(auc)
        print(f"Fold {fold+1} AUC: {auc:.6f}\n")

    oof_auc = roc_auc_score(y, oof_preds)
    print("─" * 45)
    for i, s in enumerate(fold_aucs):
        print(f"  Fold {i+1}: {s:.6f}")
    print("─" * 45)
    print(f"  Mean  : {np.mean(fold_aucs):.6f} ± {np.std(fold_aucs):.6f}")
    print(f"  OOF   : {oof_auc:.6f}")
    print("─" * 45)

    return oof_preds, test_preds, fold_aucs, oof_auc, model

oof_preds, test_preds, fold_aucs, oof_auc, last_model = xgb_oof(
    X, y, X_test, best_params, CONFIG
)

## 10. Submission

In [ ]:
submission = pd.DataFrame({
    "id"        : test["id"],
    "PitNextLap": test_preds,
})
submission.to_csv("submission.csv", index=False)

print("✅ Submission saved")
print(f"   Shape  : {submission.shape}")
print(f"   Min    : {test_preds.min():.4f}")
print(f"   Max    : {test_preds.max():.4f}")
print(f"   Mean   : {test_preds.mean():.4f}")
print(f"\n   OOF AUC: {oof_auc:.6f}")
print(submission.head())

## Summary

| | Score |
|---|---|
| OOF ROC-AUC | 0.957087 |
| Public LB | 0.94925 |

Ways to push this further:
- 🛞 Advanced tyre lifecycle modeling
- 🕛 Rolling lap time features
- 🧠 Ensembles → XGBoost + CatBoost + RealMLP
- 🔥 Interaction features beyond TyreLife × Compound

---
*Upvote if useful 🙌*  

**Checkout CatBoost for ensembling [here](https://www.kaggle.com/code/simarbirsinghsandhu/catboost-gpu-fe-encoding)**

*Related: [Detailed EDA + Original Dataset Analysis](https://www.kaggle.com/code/simarbirsinghsandhu/detailed-eda-on-comp-original-dataset)*